In [ ]:
import os
import subprocess

# Definisci il percorso esatto della tua repo su Drive
REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"

# ==========================================
# 0. AUTO-UPDATE REPOSITORY (COLAB ONLY)
# ==========================================
try:
    import google.colab
    print("Colab environment detected. Pulling latest code from GitHub...")

    google.colab.drive.mount('/content/drive', force_remount=True)
    
    # Assicurati che la cartella esista prima di provare a fare pull
    if os.path.exists(REPO_DIR):
        # Il parametro cwd=REPO_DIR dice a subprocess di eseguire il comando LÌ dentro
        result = subprocess.run(
            ["git", "pull", "origin", "main"], 
            cwd=REPO_DIR, 
            capture_output=True, 
            text=True
        )
        
        if result.returncode == 0:
            print(f"Git Pull Success:\n{result.stdout.strip()}")
        else:
            print(f"Git Pull Failed:\n{result.stderr.strip()}")
            
        # Sposta anche l'ambiente Python nella cartella per le successive importazioni locali
        os.chdir(REPO_DIR)
        
    else:
        print(f"Errore: La cartella {REPO_DIR} non esiste su Drive. Fai prima il setup base.")

except ImportError:
    print("Local environment detected. Skipping auto-pull to protect your uncommitted work.")

In [ ]:
import os
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

hf_token = None

# ==========================================
# 1. SMART TOKEN LOADING
# ==========================================
try:
    # Attempt to load Colab-specific modules
    from google.colab import userdata
    print("Colab environment detected. Fetching token from Secrets...")
    hf_token = userdata.get('HF_TOKEN')

except ImportError:
    # If it fails, we are running locally in Antigravity/VS Code
    print("Local environment detected. Reading token from .env file...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    # Extract the token, stripping whitespace and quotes
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("No .env file found in the local directory.")

# ==========================================
# 2. AUTHENTICATION
# ==========================================
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Hugging Face login successful!")
else:
    print("Warning: HF_TOKEN not found! Model download will fail if it's gated.")

# ==========================================
# 3. MODEL LOADING
# ==========================================
model_id = "meta-llama/Llama-4-Scout-17B-16E" 
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16
)
print("Model loaded successfully!")